# NB02: Data Transformation

**LSE ME204 – Data Engineering Principles for the Social Sciences (2026)**

**LSE ID:** 250104200

</div>

## Setup
Import packages

In [2]:
import pandas as pd
import datetime as dt
import json
import os

Load constants

In [3]:
START_YEAR = 2019
END_YEAR = 2026
EMP_SERIES = "CES" # National Employment, Hours, and Earnings; seasonally adjusted
EMP_DATA_TYPE = "01" # Employement (Level 1000s)

Function to load all json files in raw data folder for a given indicator

In [19]:
def load_data(indicator):
    '''
    Load all json files within the raw data folder that matches the indicator passed to the function
    '''
    all_files = os.listdir(f"../data/raw/{indicator}")
    print(all_files)
    all_json_files = []

    for file in all_files:
            if os.path.isfile(f"../data/raw/{indicator}/{file}") and file.endswith(".json"):
                file_path = f"../data/raw/{indicator}/{file}"
                print(f"Found {file_path}")

                with open(file_path, mode="r", encoding="utf-8") as f:
                    raw_data = json.load(f)
                    # all_json_files.append(file)

                df = (pd.json_normalize(
                    raw_data["Results"]["series"], 
                    record_path="data",
                    meta="seriesID")
                )
                all_json_files.append(df)
    
    full_df = pd.concat(all_json_files)
    return full_df

raw_df = load_data("employment")



['bls_employment_4.json', 'bls_employment_3.json', 'bls_employment_2.json', 'bls_employment_1.json']
Found ../data/raw/employment/bls_employment_4.json
Found ../data/raw/employment/bls_employment_3.json
Found ../data/raw/employment/bls_employment_2.json
Found ../data/raw/employment/bls_employment_1.json


Define a function to clean the raw dataframe - allows me to repeat on other BLS series I may want to add to the analysis

In [34]:
def clean_bls(df):
    '''
    Clean the dataframe read from raw BLS response json by formatting the 
    year and period columns as datetime and dropping unneccessary columns.
    '''

    df = df.assign(
        month=df["period"].str[1:].astype(int),
        day=1
    )
    df["date"] = pd.to_datetime(df[["year","month","day"]])

    df["ce_code"] = df["seriesID"].str[3:11]

    df = df[["date", "ce_code", "value"]]
    return df

Re-map industry name and NAICS code onto the series ID using the `aiie_bls_map`

In [38]:
clean_df = clean_bls(raw_df)

aiie_bls_map = pd.read_csv("../data/reference/aiie_bls_map.csv")
aiie_bls_map["ce_code"] = aiie_bls_map["ce_code"].astype(str)

tidy_df = pd.merge(
    left=clean_df,
    right=aiie_bls_map,
    on="ce_code",
    how="left"
)
tidy_df = tidy_df[["date", "industry_name", "aiie", "value"]]
tidy_df = tidy_df.rename({"aiie":"ai_exposure","value":"employment"}, axis=1)
tidy_df

,date,industry_name,ai_exposure,employment
0,2026-06-01,Offices of physicians,1.009907,3053.7
1,2026-05-01,Offices of physicians,1.009907,3054.6
2,2026-04-01,Offices of physicians,1.009907,3050.7
3,2026-03-01,Offices of physicians,1.009907,3049.8
4,2026-02-01,Offices of physicians,1.009907,3011.8
...,...,...,...,...
17046,2019-05-01,Architectural and structural metals manufacturing,-0.500582,398.9
17047,2019-04-01,Architectural and structural metals manufacturing,-0.500582,398.0
17048,2019-03-01,Architectural and structural metals manufacturing,-0.500582,396.8
17049,2019-02-01,Architectural and structural metals manufacturing,-0.500582,396.3


In [39]:
tidy_df.to_csv("../data/processed/employment.csv", index=False)